# Agents, Tools, MCPs and All That: Part 2
---------

## 1. Introduction
-------------------

In [my last post](https://michael-harmon.com/posts/mcp1/) I went over how to create agents with [LangChain](https://www.langchain.com/) and give them access to tools through a local MCP server using [FastMCP](https://gofastmcp.com/getting-started/welcome). In this post I want to go over how to deploy this MCP server through a CI/CD pipeline using [Terraform](https://www.hashicorp.com/en/products/terraform), [GitHub Actions](https://github.com/features/actions) and [Google Cloud Run](https://cloud.google.com/run?hl=en). I'll show how to use the remote MCP server with [LangChain](https://www.langchain.com/) agent like I did previousl. Lastly, I'll also show how to use the server with [Google AntiGravity](https://antigravity.google/) agent. Setting up a remote MCP server is important since it allows people (or agents) anywhere in the world use you the tools.

Let's get into it!

## 2. CI/CD With Terraform & GitHub Actions
-------------------

In a [prior post](https://michael-harmon.com/posts/chatbot2/) I covered how to use [GitHub Actions](https://github.com/features/actions) to create a [CI/CD](https://en.wikipedia.org/wiki/CI/CD) pipeline to deploy a containerized application to [Google Cloud Run](https://cloud.google.com/run?hl=en). Google Cloud Run is serverless which is a great option for our MCP server because the server itself does not have any state. Instead each of the tools calls APIs or databases ([MongoDB](https://www.mongodb.com/) and [Redis](https://redis.io/)) and just requires access to them by passing my credentials into the [Docker container](https://www.docker.com). I deployed my [MCP server](https://github.com/mdh266/langchain-fastmcp/blob/main/mymcp/server.py) using a GitHub actions yaml file called [build-image.yml](https://github.com/mdh266/langchain-fastmcp/blob/main/.github/workflows/build-image.yml). This job builds a Docker image, pushes it to DockerHub, and then deploys it to Google Cloud Run. All my credentials (API Keys, Database usernames/passwords) are stored in [GitHub secrets](https://docs.github.com/en/actions/how-tos/write-workflows/choose-what-workflows-do/use-secrets) and are [pushed to the container](https://github.com/mdh266/langchain-fastmcp/blob/main/.github/workflows/build-image.yml#L54) during deployment to Cloud Run. 

**It is important to not we do not pass the secrets when [building the image](https://github.com/mdh266/langchain-fastmcp/blob/main/.github/workflows/build-image.yml#L34) as that would make our credentials accessible to anyone that pulls the image!** I covered a lot of the details in this CI/CD pipeline during in a prior post so I won't repeat it here. The one new thing I will mention is that I added [this line](https://github.com/mdh266/langchain-fastmcp/blob/main/.github/workflows/build-image.yml#L18) to the job to allow me to have "Run Workflow" on the GitHub Actions page so that I can trigger the build without pushing code changes.

Now to discuss the new [GitHub Action job](https://github.com/mdh266/langchain-fastmcp/blob/main/.github/workflows/terraform.yaml) that I will talk about is about the Terraform. Terraform is declarative language used to create [infrastructure as code](https://en.wikipedia.org/wiki/Infrastructure_as_code). This means you declare what you need and Terraform figures out how to create it. 

The terraform code is located in the `terraform` directory. It should contain a file called [main.tf](https://github.com/mdh266/langchain-fastmcp/blob/main/terraform/main.tf) which is a module that has declares the provider we use. The first part is,



```terraform
terraform {
  required_version = ">=1.15.3"

  required_providers {
    google = {
      source  = "hashicorp/google"
      version = "~>5.0"
    }
  }
}

provider "google" {
  project = var.project_id
  region  = var.region
}
```

This block of code tells Terraform we will be using Google cloud provider with specificed `project` in the region `region`. The `var.project_id` signifies these are variable values. Theses variables are described in the [variables.tf](https://github.com/mdh266/langchain-fastmcp/blob/main/terraform/variables.tf) files. The format for variables is,


```terraform
variable <variable-name> {
  description = ...
  type        = ...
  sensitive   = true or false
}
```

The non-sensitive variables will have their values the [terraform.tfvars](https://github.com/mdh266/langchain-fastmcp/blob/main/terraform/terraform.tfvars) file and add to the repo. Sensitive variables will have their values supplied as secrets during CI/CD, they should not be stored in the code.

Next in the `main.tf` file we will talk about resources in Terraform. Each resource has the basic form

```terraform
resource "<cloud-provider-service-name>" "<service-name>" {
  name = ...
  region = ...
  
  ....
}

The Terraform code to create a Google Cloud Run Service is,

```terraform
resource "google_cloud_run_service" "mcp_service" {
  name     = var.service_name
  location = var.region

  template {
    spec {
      containers {
        image = var.container_image

        resources {
          limits = {
            cpu    = "1"
            memory = "256Mi"
          }
        }
      }
    }
  }
}
```

Note that this pulls the container image wich is what we deployed in the [build-iamge.yaml](https://github.com/mdh266/langchain-fastmcp/blob/main/.github/workflows/build-image.yml) GitHub Action job. The resource limits the container to use 1 CPU and a maximum 256MB of RAM. This is all thats needed to create the Cloud Run Service! However, we have not given anyone access to it yet. To do so we must create the IAM service,

```terraform
resource "google_cloud_run_service_iam_member" "public_invoker" {
  location = google_cloud_run_service.mcp_service.location
  project  = var.project_id
  service  = google_cloud_run_service.mcp_service.name

  role   = "roles/run.invoker"
  member = "allUsers"
}
```

Notice that the location is the same location as the `mcp_service`, they should be co-located! The service we are giving access to is the name of the `mcp_service`. Lastly, notice we give this a IAM role the ability of all users to invoke it with the `role` and `member` values. 

The last thing we need to discuss is [outputs.tf](https://github.com/mdh266/langchain-fastmcp/blob/main/terraform/outputs.tf)

```terraform
output "service_url" {
  description = "Public URL of the Cloud Run service"
  value       = google_cloud_run_service.mcp_service.status[0].url
}
```

This is a simple definition of the return of the Terraform build. It returns the url of the Google Cloud Run MCP end point.

Now we `cd` into `terraform` directory and say,

    terraform init

<img src="https://github.com/mdh266/langchain-fastmcp/blob/main/images/terraform1.png?raw=1">

This will createa a `.terraform` directory along with a file `.terraform.lock.hcl`.  

Next we run,

    terraform plan -var-file=terraform.tfvars

this creates a plan for the infrastructure to build using the variable values from the [terraform.tfvars](https://github.com/mdh266/langchain-fastmcp/blob/main/terraform/terraform.tfvars) which uses the non-sensitive variables. The plan is printed to the screen in the above, but to print it to a file `ffplan` we use

    terraform plan -var-file=terraform.tfvars -out=tfplan

Terraform will prompt you for the sensitive variable values and create the infrastructure plan as shown below,

<img src="https://github.com/mdh266/langchain-fastmcp/blob/main/images/terraform2.png?raw=1">

To build the actual infrastructure from the plan `tfplan` we run, 

    terraform apply -auto-approve tfplan

After the infrastructure is built terraform will rhe return value will be the url for the end-point! The end-point will look like `https://agentsmcp-service-{mcp_num}.us-central1.run.app`. The `agentsmcp-service` and the `us-central1` came from the `service_name` and `region` in the `terraform.tfvars` respectively.

To build this infrastructure as part of a CI/CD build I created [terraform.yaml](https://github.com/mdh266/langchain-fastmcp/blob/main/.github/workflows/terraform.yaml) GitHub Actions. The things I'll call out [on this line](https://github.com/mdh266/langchain-fastmcp/blob/main/.github/workflows/terraform.yaml#L6) is that we first set the build to happen any time yaml file or any of in `terraform` folder change. The other thing I will point out is that the project_id is set as [an environment variable](https://github.com/mdh266/langchain-fastmcp/blob/main/.github/workflows/terraform.yaml#L17) that begins with `TF_VAR_` to signify its a Terraform variable. The rest of the yaml file is standard and the steps in the job follow the sequence of the commands we discussed. 



## 3. Using A Remote MCP Server With An Agent
-------------------

Now that we have a remote MCP server we can connect an agent to it just the same way did in the last post. In order to make a LangChain agent have access to the tools supplied by an MCP server we need to use a specialized MCP client for LangChain called [LangChain MCP Adapter](https://reference.langchain.com/python/langchain-mcp-adapters). This can be imported and set to the address of the MCP server and set transport type to HTTP:

In [ ]:
from dotenv import load_dotenv, find_dotenv
import os
from langchain_mcp_adapters.client import MultiServerMCPClient  
load_dotenv(find_dotenv())

mcp_num = os.environ.get("MCP_NUM")
client = MultiServerMCPClient({"config": 
                        {"url": f"https://agentsmcp-service-{mcp_num}.us-central1.run.app/mcp", 
                         "transport": "http"}
})

Now we can get the tools:

In [22]:
tool_list = await client.get_tools()
tool_list

[StructuredTool(name='get_weather', description='Fetch current weather for *city* from OpenWeatherMap.\n\nThe function reads the API key from the ``OPEN_WEATHER_MAP_API_KEY``\nenvironment variable unless an explicit ``api_key`` argument is supplied.', args_schema={'properties': {'city': {'type': 'string', 'description': 'City name to query, e.g. ``"London"``.'}, 'api_key': {'anyOf': [{'type': 'string'}, {'type': 'null'}], 'default': None, 'description': 'Optional explicit API key; if omitted the environment variable\nis used.'}}, 'required': ['city'], 'type': 'object', 'additionalProperties': False}, metadata={'_meta': {'fastmcp': {'tags': []}}}, handle_tool_error=<function _handle_mcp_tool_error at 0x119f9bce0>, response_format='content_and_artifact', coroutine=<function convert_mcp_tool_to_langchain_tool.<locals>.call_tool at 0x11c0bc540>),
 StructuredTool(name='convert_address_to_point', description='Convert a street address to a Shapely Point object.', args_schema={'properties': {'

Notice these are [LangChain tools](https://docs.langchain.com/oss/python/langchain/tools)! The adapter converted the [FastMCP](https://gofastmcp.com/getting-started/welcome) tool to a LangChain one! Now let's create an agent and give it access to the tools from the MCP server,

In [23]:
from langchain_groq import ChatGroq
from langchain.agents import create_agent

model = ChatGroq(model="openai/gpt-oss-120b")

agent = create_agent(model=model, tools=tool_list)

result = await agent.ainvoke({
    'messages': [{'role': 'user', 
                 'content': 'What police precinct is 306 West 54th Street, manhattana, manhattan in? Get me their info.'
}]})

In [24]:
print(result.get("messages")[-1].content)

**NYC Police Precinct:** **Precinct 18**  

**Details**

| Item | Information |
|------|--------------|
| **Borough** | Manhattan |
| **Address** | 306 West 54th Street, New York, NY 10019 |
| **Phone** | 212‑767‑8400 |
| **Coverage Area** | The 18th Precinct serves the Midtown West area of Manhattan, including the neighborhoods around Hell’s Kitchen, the Theater District, and the Upper West Side near the address you provided. |

If you need to contact the precinct, you can call the non‑emergency number above (or dial **911** for emergencies).


It worked!

## 4. Using A Remote MCP Server With Antigravity
-----------------------------------

## 5. Conclusions
--------------------


<!--  -->